# MobileNet — Etapa 1: Tratamento de Dados

Este notebook cobre todas as etapas de pré-processamento do dataset de classificação de faixa de preço de celulares, desde a análise exploratória até a preparação dos dados para o treinamento do MLP.

---

**Variável alvo:** `faixa_preco`

| Classe | Descrição        |
|--------|------------------|
| 0      | Baixo custo      |
| 1      | Custo médio      |
| 2      | Alto custo       |
| 3      | Custo muito alto |

## 1. Importações

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Exibe todas as colunas do DataFrame sem truncamento
pd.set_option('display.max_columns', None)

# Define o tema visual dos gráficos
sns.set_theme(style='whitegrid')

# Diretório de saída para as imagens geradas nesta etapa
IMAGES_DIR = '../images/eda'
os.makedirs(IMAGES_DIR, exist_ok=True)

---
## 2. Carregamento dos Dados

In [ ]:
# Leitura dos arquivos CSV de treino e teste
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Treino: {train.shape[0]} amostras, {train.shape[1]} colunas')
print(f'Teste:  {test.shape[0]} amostras, {test.shape[1]} colunas')

# Visualização das primeiras linhas do conjunto de treino
train.head()

---
## 3. Análise Exploratória (EDA)

### 3.1 Tipos de dados e valores ausentes

O primeiro passo é verificar os tipos de cada coluna e a presença de valores ausentes. Valores ausentes podem comprometer o treinamento do modelo se não forem tratados adequadamente.

In [ ]:
# Exibe informações gerais: tipos, contagem de não-nulos e uso de memória
train.info()

In [ ]:
# Contagem de valores ausentes por coluna
ausentes = train.isnull().sum()

print('Valores ausentes por coluna:')
print(ausentes[ausentes > 0] if ausentes.any() else 'Nenhum valor ausente encontrado.')

### 3.2 Estatísticas descritivas

As estatísticas descritivas fornecem um resumo quantitativo de cada feature:

- **Média:** $\bar{x} = \dfrac{1}{n}\displaystyle\sum_{i=1}^{n} x_i$

- **Desvio padrão:** $s = \sqrt{\dfrac{1}{n-1}\displaystyle\sum_{i=1}^{n}(x_i - \bar{x})^2}$

- **Quartis e amplitude interquartil (IQR):** $\text{IQR} = Q_3 - Q_1$ — valores muito acima de $Q_3 + 1{,}5 \cdot \text{IQR}$ ou abaixo de $Q_1 - 1{,}5 \cdot \text{IQR}$ são considerados outliers.

In [ ]:
# Estatísticas descritivas de todas as features numéricas
train.describe().round(2)

### 3.3 Distribuição da variável alvo

Um dataset balanceado garante que o modelo aprenda igualmente bem para todas as classes. O desequilíbrio entre classes pode levar o modelo a favorecer as mais frequentes, reduzindo a performance nas demais.

In [ ]:
# Contagem de amostras por classe, ordenada pelo índice
contagem = train['faixa_preco'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))

# Gráfico de barras com paleta de azuis
bars = ax.bar(contagem.index, contagem.values, color=sns.color_palette('Blues_d', 4))

# Rótulos de valor no topo de cada barra
ax.bar_label(bars, padding=4, fontsize=11)

ax.set_title('Distribuição da Variável Alvo (faixa_preco)', fontsize=13)
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['0 – Baixo', '1 – Médio', '2 – Alto', '3 – Muito alto'])

plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/01_target_distribution.png', dpi=150)
plt.show()

# Proporção percentual de cada classe
print('\nProporção por classe:')
print((contagem / len(train) * 100).round(2).astype(str) + '%')

### 3.4 Correlação entre features

O **coeficiente de correlação de Pearson** mede a intensidade e direção da relação linear entre duas variáveis contínuas:

$$r_{xy} = \frac{\displaystyle\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\displaystyle\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \displaystyle\sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

| Valor de $r$ | Interpretação               |
|--------------|-----------------------------|
| $r = 1$      | Correlação positiva perfeita |
| $r = -1$     | Correlação negativa perfeita |
| $r = 0$      | Sem correlação linear        |

Features com alta correlação entre si podem introduzir **redundância** no modelo, enquanto features com alta correlação com a variável alvo são as mais importantes para a classificação.

In [ ]:
# Mapa de calor da matriz de correlação de Pearson
fig, ax = plt.subplots(figsize=(15, 11))
sns.heatmap(
    train.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Mapa de Correlação de Pearson', fontsize=14)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/02_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Correlação de cada feature com a variável alvo, ordenada por valor absoluto
corr_target = train.corr()['faixa_preco'].drop('faixa_preco').sort_values(key=abs, ascending=False)

print('Correlação com faixa_preco (ordenada por |r|):')
print(corr_target.round(3).to_string())

### 3.5 Distribuição das features por classe

Os **boxplots** permitem comparar a distribuição de uma feature entre as classes da variável alvo. Features com distribuições bem separadas entre as classes têm maior **poder discriminativo** e serão mais úteis ao modelo.

Cada boxplot exibe:
- A **mediana** (linha central)
- O **IQR** (caixa: $Q_1$ a $Q_3$)
- Os **whiskers** ($1{,}5 \times \text{IQR}$)
- Os **outliers** (pontos além dos whiskers)

In [ ]:
# Seleciona as 6 features mais correlacionadas com a variável alvo
features_destaque = corr_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Um boxplot por feature, separado por classe
for i, feat in enumerate(features_destaque):
    sns.boxplot(data=train, x='faixa_preco', y=feat, ax=axes[i], palette='coolwarm')
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel('Classe')

fig.suptitle('Top 6 Features — Distribuição por Classe', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/03_features_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Separação de Features e Variável Alvo

In [ ]:
# Separa as features (X) da variável alvo (y)
X = train.drop(columns='faixa_preco')
y = train['faixa_preco']

print(f'Features (X): {X.shape}')
print(f'Classes  (y): {sorted(y.unique())}')

---
## 5. Normalização — StandardScaler

O MLP é sensível à escala das features: features em escalas muito diferentes podem dominar o gradiente e dificultar a convergência. O **StandardScaler** padroniza cada feature para ter média zero e desvio padrão unitário:

$$z = \frac{x - \mu}{\sigma}$$

Onde:
- $x$ — valor original da feature
- $\mu$ — média da feature calculada no conjunto de treino
- $\sigma$ — desvio padrão da feature calculado no conjunto de treino
- $z$ — valor padronizado (resultado)

> **Atenção ao vazamento de dados (*data leakage*):** o scaler deve ser ajustado (`fit`) **apenas nos dados de treino**. Aplicá-lo também nos dados de validação/teste com `transform` garante que nenhuma informação do futuro vaze para o modelo durante o treinamento.

In [ ]:
# Divisão treino/validação antes da normalização para evitar data leakage
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,       # 20% para validação
    random_state=42,     # semente para reprodutibilidade
    stratify=y           # mantém a proporção das classes nos dois conjuntos
)

# Ajusta o scaler apenas no treino e aplica em ambos os conjuntos
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)

print(f'Treino:    {X_train.shape}')
print(f'Validação: {X_val.shape}')

In [ ]:
# Verificação da normalização: média deve ser ≈ 0 e desvio padrão ≈ 1
df_verificacao = pd.DataFrame(X_train, columns=X.columns)

print('Média das primeiras 5 features (esperado: ≈ 0):')
print(df_verificacao.mean().head().round(6).to_string())

print('\nDesvio padrão das primeiras 5 features (esperado: ≈ 1):')
print(df_verificacao.std().head().round(6).to_string())

---
## 6. Verificação da Divisão Treino / Validação

Com a divisão **80% treino / 20% validação** e a opção `stratify=y`, garantimos que a proporção de cada classe seja preservada em ambos os conjuntos. Isso é fundamental para que a métrica de validação seja representativa do desempenho real do modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Gráfico de barras para treino e validação lado a lado
for ax, (nome, serie) in zip(axes, [('Treino', y_train), ('Validação', y_val)]):
    counts = serie.value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette('Blues_d', 4))
    ax.bar_label(bars, padding=3, fontsize=10)
    ax.set_title(f'Distribuição — {nome}', fontsize=12)
    ax.set_xlabel('Classe')
    ax.set_ylabel('Quantidade')
    ax.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/04_train_val_split.png', dpi=150)
plt.show()

---
## 7. Resumo do Pré-processamento

| Etapa                    | Detalhe                                     |
|--------------------------|---------------------------------------------|
| Valores ausentes         | Nenhum                                      |
| Tipos de dados           | Todos numéricos (int64 / float64)           |
| Normalização             | StandardScaler ($\mu=0$, $\sigma=1$)        |
| Divisão                  | 80% treino / 20% validação (stratify=y)     |
| Amostras de treino       | 1600                                        |
| Amostras de validação    | 400                                         |
| Número de features       | 20                                          |
| Número de classes        | 4                                           |

Os dados estão preparados para o treinamento do MLP na **Etapa 2**.